# 01 · Framing the task, and how not to fool yourself

**Goal:** turn raw time series into supervised forecasting examples, split them correctly, and
*see* what goes wrong with the wrong split.

**Time:** ~45 min · **You'll write:** `windows_for_patient`, `stratified_subject_split`,
`nearest_neighbour_forecast`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

from biotwin.data.dataset import Cohort, make_windows, standard_sets, subset, LH, H, STEPS_PER_DAY
cohort = Cohort.load("data/cohort.npz")
print("history LH =", LH, "steps (6 h); horizon H =", H, "steps (60 min)")

## 1. The task

At time $t_0$, the model sees:
- CGM over $[t_0 - L_H,\ t_0)$: the last 6 h;
- insulin and carbs over $[t_0 - L_H,\ t_0 + H)$: the past **and the planned next hour**;

and predicts CGM over $[t_0,\ t_0 + H)$, which is 5…60 min ahead.

Feeding in the *planned* inputs is what makes this a twin: we can change them and ask "what if".

## Exercise 1 — build windows

For one patient, produce every window whose **first target index** $t_0$ satisfies
$\max(t_{lo}, L_H) \le t_0 \le t_{hi} - H$. Return a dict of tensors:
`cgm_hist [N, LH]`, `ins [N, LH+H]`, `carbs [N, LH+H]`, `y [N, H]`, `t0 [N]`.


<details><summary><b>Hint</b></summary>

Loop `t0` over the range and slice. Collect lists, then `torch.as_tensor(np.array(list))`. The `max(t_lo, LH)` makes sure there are always `LH` steps of history.

</details>

In [ ]:
def windows_for_patient(cgm, ins, carbs, t_lo, t_hi, LH, H):
    # TODO
    raise NotImplementedError

def check(f):
    name = "adult#003"; p = cohort.idx(name)
    got = f(cohort.cgm[p], cohort.insulin[p], cohort.carbs[p], 0, 600, LH, H)
    ref = make_windows(cohort, [name], (0, 600))
    for k in ["cgm_hist", "ins", "carbs", "y", "t0"]:
        close(got[k], ref[k], what=k)

windows_for_patient = use(windows_for_patient, sol.windows_for_patient, check)

Consecutive windows overlap almost completely: two neighbours share 71 of their 72 history
samples. Keep that in mind for the next sections.

## Exercise 2 — split by patient

The question we care about is "does it work on a **new person**?", so train, validation and test
must contain **different patients**. Stratify by age group (`adolescent`, `adult`, `child`): from
each group, after a seeded permutation of its sorted members, the first `n_test` go to test, the
next `n_val` to val, and the rest to train.


<details><summary><b>Hint</b></summary>

`rng = np.random.default_rng(seed)`; group with `name.split('#')[0]`; `rng.permutation(sorted(members))`.

</details>

In [ ]:
def stratified_subject_split(names, n_val, n_test, seed):
    # TODO: return {"train": [...], "val": [...], "test": [...]}
    raise NotImplementedError

def check(f):
    names = [str(x) for x in cohort.names]
    sp = f(names, 2, 2, 0)
    a, b, c = set(sp["train"]), set(sp["val"]), set(sp["test"])
    assert not (a & b or a & c or b & c), "splits overlap"
    assert a | b | c == set(names), "some patients are missing"
    for g in ["adolescent", "adult", "child"]:
        assert sum(x.startswith(g) for x in c) == 2 and sum(x.startswith(g) for x in b) == 2, f"group {g} not stratified"
    assert f(names, 2, 2, 0) == sp, "same seed must give the same split"

stratified_subject_split = use(stratified_subject_split, sol.stratified_subject_split, check)
print(stratified_subject_split([str(x) for x in cohort.names], 2, 2, 0)["test"])

## Exercise 3 — a memoriser

To show why the split matters, we need a model that is very good at memorising: **1-nearest
neighbour**. For each test window, find the closest training window (Euclidean distance on its
features) and copy its target.


<details><summary><b>Hint</b></summary>

`torch.cdist(test_x, train_x)` gives all pairwise distances; `.argmin(1)` picks the neighbour.

</details>

In [ ]:
def nearest_neighbour_forecast(train_x, train_y, test_x):
    # TODO: return train_y of the nearest training row, for every test row
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    tx, ty, qx = torch.randn(50, 4, generator=g), torch.randn(50, 3, generator=g), torch.randn(7, 4, generator=g)
    brute = torch.stack([ty[((tx - q) ** 2).sum(1).argmin()] for q in qx])
    close(f(tx, ty, qx), brute, what="1-NN")

nearest_neighbour_forecast = use(nearest_neighbour_forecast, sol.nearest_neighbour_forecast, check)

## 2. The experiment: random windows vs. new patients

Same model, same features, two ways of splitting:
- **A, random windows:** pool all windows of 6 patients and hold out a random 20 %.
- **B, new patients:** train on the 18 training patients, test on 6 unseen patients.

In [ ]:
S = standard_sets(cohort, seed=0)
feat = lambda w: torch.cat([w["cgm_hist"] - w["cgm_hist"][:, -1:], 10 * w["ins"][:, LH:], w["carbs"][:, LH:]], 1)
delta = lambda w: w["y"] - w["cgm_hist"][:, -1:]

# A: random-window split inside the same 6 patients
allw = make_windows(cohort, S["split"]["test"])
perm = torch.randperm(len(allw["y"]), generator=torch.Generator().manual_seed(0))
te_i, tr_i = perm[: len(perm) // 5], perm[len(perm) // 5:]
A_tr, A_te = subset(allw, tr_i), subset(allw, te_i)
pA = nearest_neighbour_forecast(feat(A_tr), delta(A_tr), feat(A_te)) + A_te["cgm_hist"][:, -1:]

# B: patient split
B_tr, B_te = S["train"], S["test_eval"]
pB = nearest_neighbour_forecast(feat(B_tr), delta(B_tr), feat(B_te)) + B_te["cgm_hist"][:, -1:]

rm = lambda p, w: (p[:, -1] - w["y"][:, -1]).pow(2).mean().sqrt().item()
print(f"1-NN, random-window split : RMSE 60 min = {rm(pA, A_te):5.1f} mg/dL")
print(f"1-NN, patient split       : RMSE 60 min = {rm(pB, B_te):5.1f} mg/dL")
print("(for scale: a GRU trained properly gets ~32 on the patient split)")

With the random split, the memoriser looks better than any real model in this project. Its
"neighbour" is usually the same patient's window shifted by 5 minutes, which has almost the same
answer. With a patient split it's useless.

**Rule:** split along the axis you want to generalise over. Here that's patients. Within a test
patient, we also split **in time** (days 0–4 adapt, day 5 calibrate, days 6–9 evaluate), because
in deployment the future comes after the past.

In [ ]:
for k in ["train", "val", "test_fit", "test_cal", "test_eval"]:
    w = S[k]
    days = sorted(set((w["t0"] // STEPS_PER_DAY).tolist()))
    print(f"{k:9s} {len(w['y']):6d} windows, target days {days[0]}–{days[-1]}")

## Quiz

Answer in your own words before opening the answers.

**1. Why does the model get *future* insulin and carbs but not future CGM?**

<details><summary>Answer</summary>

Future insulin and carbs are decisions (planned inputs). Asking 'what if I eat X' means changing them. Future CGM is what we predict. Giving it to the model, even partly, is leakage.

</details>

**2. Two adjacent windows share 71 of 72 history samples. What does that do to a random train/test split?**

<details><summary>Answer</summary>

Almost every test window has a near-duplicate in training, so test error measures interpolation between neighbours, not generalisation. The 1-NN experiment showed how much that flatters a model.

</details>

**3. Why stratify the patient split by age group?**

<details><summary>Answer</summary>

With only 30 patients, a random split could put no children in the test set, for example. Children behave differently (larger swings), so the score would depend on luck. Stratifying keeps each split representative.

</details>

**4. Inside a test patient, why adapt on days 0–4 and evaluate on days 6–9, and not a random choice of days?**

<details><summary>Answer</summary>

In deployment you adapt on the past and predict the future. Using later days to adapt and earlier days to test would leak future information (e.g. a drift in the patient). Day 5 sits between them for calibration, so interval widths aren't tuned on the evaluation data.

</details>

**5. You get 15 mg/dL RMSE at 60 min. What three checks would you make before believing it?**

<details><summary>Answer</summary>

1) Is the split by patient (and by time within a patient)? 2) Is any target-derived signal in the inputs (future CGM, the noise-free bg, statistics computed over the whole series)? 3) Compare against persistence and the noise floor on exactly the same windows. A leakage unit test like `tests/test_dataset.py` helps.

</details>